In [1]:
import json
import os
import re
import subprocess
from pathlib import Path
from urllib.parse import urlparse

try:
    import pandas as pd
    PANDAS_AVAILABLE = True
except ImportError:
    PANDAS_AVAILABLE = False

In [2]:
class MaintainableSecurityChecker:
    """
    MaintainableSecurityChecker

    This checker evaluates whether a research software artifact supports
    ongoing security maintenance over time.

    It checks for:
    - SECURITY.md or vulnerability reporting policy
    - dependency update automation
    - security scanning workflows
    - CI/CD workflows
    - tests
    - issue templates
    - contribution guidelines
    - changelog / release notes
    - dependency files and lock files
    - static-analysis or linting configuration

    Formal idea:
        maintainableSecurity : A → {True, False}
    """

    def __init__(
        self,
        json_file,
        download_dir="downloads",
        minimum_score=5
    ):
        self.json_file = json_file
        self.download_dir = Path(download_dir)
        self.download_dir.mkdir(parents=True, exist_ok=True)

        self.minimum_score = minimum_score
        self.artifacts = self.load_metadata(json_file)
        self.results = []

    def load_metadata(self, json_file):
        with open(json_file, "r", encoding="utf-8") as file:
            data = json.load(file)

        return data.get("artifacts", {})

    def is_git_repository(self, uri):
        return isinstance(uri, str) and uri.startswith("https://github.com/")

    def repo_name_from_uri(self, uri):
        parsed = urlparse(uri)
        repo_name = parsed.path.rstrip("/").split("/")[-1]

        if repo_name.endswith(".git"):
            repo_name = repo_name[:-4]

        return repo_name or "repository"

    def clone_repository(self, artifact_id, uri):
        repo_name = self.repo_name_from_uri(uri)
        target_dir = self.download_dir / repo_name

        if target_dir.exists():
            print(f"📁 Repository already exists: {target_dir}")
            return target_dir

        print(f"⬇️ Cloning repository: {uri}")

        try:
            result = subprocess.run(
                ["git", "clone", "--depth", "1", uri, str(target_dir)],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                timeout=120
            )

            if result.returncode != 0:
                print(f"❌ Failed to clone repository for {artifact_id}")
                print(result.stderr.strip())
                return None

            print(f"✅ Cloned to: {target_dir}")
            return target_dir

        except Exception as e:
            print(f"❌ Clone error for {artifact_id}: {e}")
            return None

    def read_text_file(self, path, max_chars=300000):
        try:
            with open(path, "r", encoding="utf-8", errors="ignore") as file:
                return file.read(max_chars)
        except Exception:
            return ""

    def find_existing_files(self, repo_dir, possible_paths):
        found = []

        for relative_path in possible_paths:
            path = repo_dir / relative_path
            if path.exists():
                found.append(relative_path)

        return found

    def find_files_by_name(self, repo_dir, names):
        found = []
        names_lower = {name.lower() for name in names}

        for root, dirs, files in os.walk(repo_dir):
            dirs[:] = [
                d for d in dirs
                if d not in {
                    ".git",
                    "__pycache__",
                    ".pytest_cache",
                    ".mypy_cache",
                    "node_modules",
                    ".venv",
                    "venv",
                    "build",
                    "dist"
                }
            ]

            for file in files:
                if file.lower() in names_lower:
                    path = Path(root) / file
                    try:
                        found.append(str(path.relative_to(repo_dir)))
                    except Exception:
                        found.append(str(path))

        return found

    def find_files_by_extensions(self, repo_dir, extensions):
        found = []

        for root, dirs, files in os.walk(repo_dir):
            dirs[:] = [
                d for d in dirs
                if d not in {
                    ".git",
                    "__pycache__",
                    ".pytest_cache",
                    ".mypy_cache",
                    "node_modules",
                    ".venv",
                    "venv",
                    "build",
                    "dist"
                }
            ]

            for file in files:
                path = Path(root) / file

                if path.suffix.lower() in extensions:
                    found.append(path)

        return found

    def detect_security_policy_files(self, repo_dir):
        return self.find_existing_files(
            repo_dir,
            [
                "SECURITY.md",
                "security.md",
                ".github/SECURITY.md",
                "docs/SECURITY.md",
                "docs/security.md"
            ]
        )

    def detect_dependency_update_files(self, repo_dir):
        return self.find_existing_files(
            repo_dir,
            [
                ".github/dependabot.yml",
                ".github/dependabot.yaml",
                "renovate.json",
                ".renovaterc",
                ".renovaterc.json",
                ".snyk"
            ]
        )

    def detect_security_workflows(self, repo_dir):
        security_files = []

        direct_paths = self.find_existing_files(
            repo_dir,
            [
                ".github/workflows/codeql.yml",
                ".github/workflows/codeql.yaml",
                ".github/workflows/security.yml",
                ".github/workflows/security.yaml",
                ".github/workflows/dependency-review.yml",
                ".github/workflows/dependency-review.yaml",
                ".github/workflows/scorecard.yml",
                ".github/workflows/scorecard.yaml"
            ]
        )

        security_files.extend(direct_paths)

        workflows_dir = repo_dir / ".github" / "workflows"

        if workflows_dir.exists():
            for path in workflows_dir.glob("*"):
                if path.suffix.lower() not in {".yml", ".yaml"}:
                    continue

                text = self.read_text_file(path)
                text_lower = text.lower()

                security_terms = [
                    "codeql",
                    "dependency-review",
                    "pip-audit",
                    "safety",
                    "bandit",
                    "semgrep",
                    "snyk",
                    "trivy",
                    "osv-scanner",
                    "scorecard",
                    "security"
                ]

                if any(term in text_lower for term in security_terms):
                    try:
                        relative = str(path.relative_to(repo_dir))
                    except Exception:
                        relative = str(path)

                    if relative not in security_files:
                        security_files.append(relative)

        return security_files

    def detect_ci_workflows(self, repo_dir):
        ci_files = []

        ci_files.extend(
            self.find_existing_files(
                repo_dir,
                [
                    ".gitlab-ci.yml",
                    "azure-pipelines.yml",
                    "Jenkinsfile",
                    "tox.ini",
                    "noxfile.py"
                ]
            )
        )

        workflows_dir = repo_dir / ".github" / "workflows"

        if workflows_dir.exists():
            for path in workflows_dir.glob("*"):
                if path.suffix.lower() in {".yml", ".yaml"}:
                    try:
                        ci_files.append(str(path.relative_to(repo_dir)))
                    except Exception:
                        ci_files.append(str(path))

        return ci_files

    def detect_test_evidence(self, repo_dir):
        test_dirs = []

        candidate_dirs = [
            "tests",
            "test",
            "testing",
            "spec",
            "specs"
        ]

        for candidate in candidate_dirs:
            path = repo_dir / candidate
            if path.exists() and path.is_dir():
                test_dirs.append(candidate)

        test_files = []

        for root, dirs, files in os.walk(repo_dir):
            dirs[:] = [
                d for d in dirs
                if d not in {
                    ".git",
                    "__pycache__",
                    ".pytest_cache",
                    ".mypy_cache",
                    "node_modules",
                    ".venv",
                    "venv",
                    "build",
                    "dist"
                }
            ]

            for file in files:
                lower = file.lower()

                if lower.startswith("test_") or lower.endswith("_test.py") or lower.endswith(".test.js"):
                    path = Path(root) / file
                    try:
                        test_files.append(str(path.relative_to(repo_dir)))
                    except Exception:
                        test_files.append(str(path))

        return test_dirs, test_files

    def detect_issue_templates(self, repo_dir):
        issue_files = []

        issue_template_dir = repo_dir / ".github" / "ISSUE_TEMPLATE"

        if issue_template_dir.exists():
            for path in issue_template_dir.glob("*"):
                if path.is_file():
                    try:
                        issue_files.append(str(path.relative_to(repo_dir)))
                    except Exception:
                        issue_files.append(str(path))

        issue_files.extend(
            self.find_existing_files(
                repo_dir,
                [
                    ".github/ISSUE_TEMPLATE.md",
                    ".github/PULL_REQUEST_TEMPLATE.md",
                    "PULL_REQUEST_TEMPLATE.md"
                ]
            )
        )

        return issue_files

    def detect_contribution_governance_files(self, repo_dir):
        return self.find_existing_files(
            repo_dir,
            [
                "CONTRIBUTING.md",
                "CONTRIBUTING.rst",
                ".github/CONTRIBUTING.md",
                "CODE_OF_CONDUCT.md",
                ".github/CODE_OF_CONDUCT.md",
                "GOVERNANCE.md",
                "MAINTAINERS.md",
                "SUPPORT.md"
            ]
        )

    def detect_release_files(self, repo_dir):
        return self.find_existing_files(
            repo_dir,
            [
                "CHANGELOG.md",
                "CHANGELOG.rst",
                "HISTORY.md",
                "RELEASE.md",
                "RELEASES.md",
                "VERSION",
                "version.txt"
            ]
        )

    def detect_dependency_files(self, repo_dir):
        return self.find_files_by_name(
            repo_dir,
            [
                "requirements.txt",
                "environment.yml",
                "environment.yaml",
                "pyproject.toml",
                "setup.py",
                "setup.cfg",
                "Pipfile",
                "package.json",
                "DESCRIPTION"
            ]
        )

    def detect_lock_files(self, repo_dir):
        return self.find_files_by_name(
            repo_dir,
            [
                "Pipfile.lock",
                "poetry.lock",
                "uv.lock",
                "requirements.lock",
                "package-lock.json",
                "yarn.lock",
                "pnpm-lock.yaml",
                "conda-lock.yml",
                "conda-lock.yaml",
                "renv.lock"
            ]
        )

    def detect_static_analysis_files(self, repo_dir):
        static_files = self.find_existing_files(
            repo_dir,
            [
                ".pre-commit-config.yaml",
                "pyproject.toml",
                "setup.cfg",
                "tox.ini",
                ".flake8",
                ".pylintrc",
                "mypy.ini",
                ".bandit",
                "bandit.yml",
                "semgrep.yml",
                ".semgrep.yml",
                ".ruff.toml",
                "ruff.toml"
            ]
        )

        found_tools = set()

        for relative in static_files:
            text = self.read_text_file(repo_dir / relative)
            lower = text.lower()

            for tool in [
                "bandit",
                "semgrep",
                "ruff",
                "flake8",
                "pylint",
                "mypy",
                "pre-commit",
                "black",
                "isort"
            ]:
                if tool in lower or tool in relative.lower():
                    found_tools.add(tool)

        return static_files, sorted(found_tools)

    def collect_security_maintenance_text(self, repo_dir):
        relevant_files = []

        names = {
            "security.md",
            "contributing.md",
            "readme.md",
            "changelog.md",
            "history.md",
            "support.md",
            "maintainers.md",
            "governance.md",
            "pyproject.toml",
            "setup.cfg",
            "tox.ini",
            ".pre-commit-config.yaml"
        }

        for root, dirs, files in os.walk(repo_dir):
            dirs[:] = [
                d for d in dirs
                if d not in {
                    ".git",
                    "__pycache__",
                    ".pytest_cache",
                    ".mypy_cache",
                    "node_modules",
                    ".venv",
                    "venv",
                    "build",
                    "dist"
                }
            ]

            for file in files:
                if file.lower() in names:
                    relevant_files.append(Path(root) / file)

        combined = ""

        for path in relevant_files[:100]:
            try:
                relative = path.relative_to(repo_dir)
            except Exception:
                relative = path

            combined += f"\n\n--- FILE: {relative} ---\n\n"
            combined += self.read_text_file(path)

        return combined

    def detect_security_maintenance_keywords(self, repo_dir):
        text = self.collect_security_maintenance_text(repo_dir)
        text_lower = text.lower()

        groups = {
            "vulnerability_reporting": [
                "vulnerability",
                "responsible disclosure",
                "security issue",
                "report a vulnerability",
                "security advisory",
                "cve"
            ],
            "dependency_updates": [
                "dependabot",
                "renovate",
                "dependency update",
                "security update",
                "upgrade dependencies"
            ],
            "remediation_workflow": [
                "patch",
                "remediation",
                "fix vulnerability",
                "security fix",
                "backport",
                "release process"
            ],
            "testing_quality": [
                "pytest",
                "unittest",
                "tox",
                "nox",
                "coverage",
                "ci",
                "continuous integration"
            ],
            "static_analysis": [
                "bandit",
                "semgrep",
                "codeql",
                "ruff",
                "pylint",
                "flake8",
                "mypy",
                "pre-commit"
            ]
        }

        findings = {}

        for group, keywords in groups.items():
            found = []

            for keyword in keywords:
                if keyword in text_lower:
                    found.append(keyword)

            findings[group] = found

        return findings

    def evaluate_maintainable_security(self, repo_dir, artifact_data):
        config = artifact_data.get("maintainable_security", {})
        minimum_score = config.get("minimum_score", self.minimum_score)

        security_policy_files = self.detect_security_policy_files(repo_dir)
        dependency_update_files = self.detect_dependency_update_files(repo_dir)
        security_workflows = self.detect_security_workflows(repo_dir)
        ci_workflows = self.detect_ci_workflows(repo_dir)
        test_dirs, test_files = self.detect_test_evidence(repo_dir)
        issue_templates = self.detect_issue_templates(repo_dir)
        governance_files = self.detect_contribution_governance_files(repo_dir)
        release_files = self.detect_release_files(repo_dir)
        dependency_files = self.detect_dependency_files(repo_dir)
        lock_files = self.detect_lock_files(repo_dir)
        static_analysis_files, static_analysis_tools = self.detect_static_analysis_files(repo_dir)
        keyword_findings = self.detect_security_maintenance_keywords(repo_dir)

        score = 0
        evidence = []
        issues = []

        if security_policy_files:
            score += 2
            evidence.append(f"Security policy found: {', '.join(security_policy_files)}")
        elif keyword_findings["vulnerability_reporting"]:
            score += 1
            evidence.append(
                f"Vulnerability reporting terms found: "
                f"{', '.join(keyword_findings['vulnerability_reporting'][:8])}"
            )
        else:
            issues.append("No SECURITY.md or vulnerability reporting process found")

        if dependency_update_files:
            score += 1
            evidence.append(f"Dependency update automation found: {', '.join(dependency_update_files)}")
        elif keyword_findings["dependency_updates"]:
            score += 1
            evidence.append(
                f"Dependency-update terms found: "
                f"{', '.join(keyword_findings['dependency_updates'][:8])}"
            )
        else:
            issues.append("No dependency update automation found")

        if security_workflows:
            score += 1
            evidence.append(f"Security scanning workflows found: {', '.join(security_workflows[:10])}")
        elif keyword_findings["static_analysis"]:
            score += 1
            evidence.append(
                f"Security/static-analysis terms found: "
                f"{', '.join(keyword_findings['static_analysis'][:8])}"
            )
        else:
            issues.append("No security scanning workflow found")

        if ci_workflows:
            score += 1
            evidence.append(f"CI/build workflows found: {', '.join(ci_workflows[:10])}")
        else:
            issues.append("No CI/build workflow found")

        if test_dirs or test_files:
            score += 1
            evidence.append(
                f"Test evidence found: dirs={len(test_dirs)}, files={len(test_files)}"
            )
        else:
            issues.append("No test suite evidence found")

        if issue_templates:
            score += 1
            evidence.append(f"Issue/PR templates found: {', '.join(issue_templates[:10])}")
        else:
            issues.append("No issue or pull-request templates found")

        if governance_files:
            score += 1
            evidence.append(f"Contribution/governance files found: {', '.join(governance_files[:10])}")
        else:
            issues.append("No contribution/governance files found")

        if release_files:
            score += 1
            evidence.append(f"Release/changelog files found: {', '.join(release_files[:10])}")
        else:
            issues.append("No changelog/release metadata found")

        if dependency_files:
            score += 1
            evidence.append(f"Dependency declaration files found: {', '.join(dependency_files[:10])}")
        else:
            issues.append("No dependency declaration files found")

        if lock_files:
            score += 1
            evidence.append(f"Lock files found: {', '.join(lock_files[:10])}")
        else:
            issues.append("No dependency lock files found")

        if static_analysis_files:
            score += 1
            evidence.append(
                f"Static-analysis/linting configuration found: "
                f"{', '.join(static_analysis_files[:10])}"
            )
        else:
            issues.append("No static-analysis/linting configuration found")

        if keyword_findings["remediation_workflow"]:
            score += 1
            evidence.append(
                f"Remediation workflow terms found: "
                f"{', '.join(keyword_findings['remediation_workflow'][:8])}"
            )
        else:
            issues.append("No remediation workflow evidence found")

        maintainable_security = score >= minimum_score

        return {
            "maintainable_security": maintainable_security,
            "score": score,
            "minimum_score": minimum_score,
            "security_policy_files": security_policy_files,
            "dependency_update_files": dependency_update_files,
            "security_workflows": security_workflows,
            "ci_workflows": ci_workflows,
            "test_dirs": test_dirs,
            "test_file_count": len(test_files),
            "issue_templates": issue_templates,
            "governance_files": governance_files,
            "release_files": release_files,
            "dependency_files": dependency_files,
            "lock_files": lock_files,
            "static_analysis_files": static_analysis_files,
            "static_analysis_tools": static_analysis_tools,
            "keyword_findings": keyword_findings,
            "evidence": evidence,
            "issues": issues
        }

    def check_artifact(self, artifact_id, artifact_data):
        title = artifact_data.get("title", "")
        uri = artifact_data.get("uri", "")

        print("\n" + "=" * 80)
        print(f"🔍 Maintainable Security Check for {artifact_id}")
        print(f"📦 Title: {title}")
        print(f"🔗 URI: {uri}")

        artifact_result = {
            "artifact_id": artifact_id,
            "title": title,
            "uri": uri,
            "maintainable_security": False,
            "status": "failed"
        }

        if not self.is_git_repository(uri):
            print("❌ Unsupported artifact type for this checker.")
            artifact_result["reason"] = "Unsupported artifact type."
            return artifact_result

        repo_dir = self.clone_repository(artifact_id, uri)

        if repo_dir is None:
            artifact_result["reason"] = "Repository could not be cloned."
            artifact_result["status"] = "not_evaluated_repository_unavailable"
            return artifact_result

        result = self.evaluate_maintainable_security(repo_dir, artifact_data)
        artifact_result.update(result)

        print("\n📊 Maintainable security evidence:")
        print(f" - Score: {result['score']} / required {result['minimum_score']}")
        print(f" - Security policy files: {', '.join(result['security_policy_files']) if result['security_policy_files'] else 'None'}")
        print(f" - Dependency update files: {', '.join(result['dependency_update_files']) if result['dependency_update_files'] else 'None'}")
        print(f" - Security workflows: {', '.join(result['security_workflows']) if result['security_workflows'] else 'None'}")
        print(f" - CI workflows: {len(result['ci_workflows'])}")
        print(f" - Test dirs: {', '.join(result['test_dirs']) if result['test_dirs'] else 'None'}")
        print(f" - Test files: {result['test_file_count']}")
        print(f" - Issue templates: {len(result['issue_templates'])}")
        print(f" - Governance files: {', '.join(result['governance_files']) if result['governance_files'] else 'None'}")
        print(f" - Release files: {', '.join(result['release_files']) if result['release_files'] else 'None'}")
        print(f" - Dependency files: {', '.join(result['dependency_files'][:8]) if result['dependency_files'] else 'None'}")
        print(f" - Lock files: {', '.join(result['lock_files'][:8]) if result['lock_files'] else 'None'}")
        print(f" - Static-analysis tools: {', '.join(result['static_analysis_tools']) if result['static_analysis_tools'] else 'None'}")

        print("\n🔎 Evidence found:")
        if result["evidence"]:
            for item in result["evidence"]:
                print(f" - {item}")
        else:
            print(" - No maintainable-security evidence found.")

        print("\n⚠️ Issues / weak evidence:")
        if result["issues"]:
            for item in result["issues"]:
                print(f" - {item}")
        else:
            print(" - No major maintainable-security issues detected.")

        if result["maintainable_security"]:
            artifact_result["status"] = "passed"
            print("\n✅ Maintainable Security Result: PASSED")
        else:
            artifact_result["status"] = "failed"
            print("\n❌ Maintainable Security Result: FAILED")

        return artifact_result

    def run(self):
        self.results = []

        print("🌱 Starting Maintainable Security Fitness Function")
        print(f"📄 Metadata file: {self.json_file}")
        print(f"📁 Download directory: {self.download_dir}")

        for artifact_id, artifact_data in self.artifacts.items():
            result = self.check_artifact(artifact_id, artifact_data)
            self.results.append(result)

        print("\n" + "=" * 80)
        print("📌 Maintainable Security Summary")
        print("=" * 80)

        for result in self.results:
            icon = "✅" if result["maintainable_security"] else "❌"
            print(f"{icon} {result['artifact_id']}: {result['status']}")

        return self.results

In [3]:
checker = MaintainableSecurityChecker(
    json_file="artifacts.json",
    download_dir="downloads",
    minimum_score=5
)

maintainable_security_results = checker.run()

🌱 Starting Maintainable Security Fitness Function
📄 Metadata file: artifacts.json
📁 Download directory: downloads

🔍 Maintainable Security Check for artifact_1
📦 Title: We provide our resources in a dedicated repository
🔗 URI: https://github.com/hihey54/hicss58
📁 Repository already exists: downloads/hicss58

📊 Maintainable security evidence:
 - Score: 1 / required 5
 - Security policy files: None
 - Dependency update files: None
 - Security workflows: None
 - CI workflows: 0
 - Test dirs: None
 - Test files: 0
 - Issue templates: 0
 - Governance files: None
 - Release files: None
 - Dependency files: requirements.txt
 - Lock files: None
 - Static-analysis tools: None

🔎 Evidence found:
 - Dependency declaration files found: requirements.txt

⚠️ Issues / weak evidence:
 - No SECURITY.md or vulnerability reporting process found
 - No dependency update automation found
 - No security scanning workflow found
 - No CI/build workflow found
 - No test suite evidence found
 - No issue or pull-

In [4]:
if PANDAS_AVAILABLE:
    df = pd.DataFrame(maintainable_security_results)

    columns_to_show = [
        "artifact_id",
        "title",
        "maintainable_security",
        "status",
        "score",
        "minimum_score",
        "security_policy_files",
        "dependency_update_files",
        "security_workflows",
        "ci_workflows",
        "test_dirs",
        "test_file_count",
        "issue_templates",
        "governance_files",
        "release_files",
        "dependency_files",
        "lock_files",
        "static_analysis_tools"
    ]

    existing_columns = [col for col in columns_to_show if col in df.columns]
    display(df[existing_columns])
else:
    for result in maintainable_security_results:
        print(result)

,artifact_id,title,maintainable_security,status,score,minimum_score,security_policy_files,dependency_update_files,security_workflows,ci_workflows,test_dirs,test_file_count,issue_templates,governance_files,release_files,dependency_files,lock_files,static_analysis_tools
0,artifact_1,We provide our resources in a dedicated reposi...,False,failed,1.0,5.0,[],[],[],[],[],0.0,[],[],[],[requirements.txt],[],[]
1,artifact_2,Trending Customer Dataset,False,not_evaluated_repository_unavailable,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,artifact_3,Python algorithms,True,passed,9.0,5.0,[],[.github/dependabot.yml],[],"[.github/workflows/project_euler.yml, .github/...",[],23.0,"[.github/ISSUE_TEMPLATE/config.yml, .github/IS...",[CONTRIBUTING.md],[],[pyproject.toml],[uv.lock],"[bandit, flake8, isort, mypy, pre-commit, pyli..."
3,artifact_4,Scikit-learn,True,passed,11.0,5.0,[SECURITY.md],[.github/dependabot.yml],"[.github/workflows/codeql.yml, .github/workflo...","[.github/workflows/publish_pypi.yml, .github/w...",[],260.0,"[.github/ISSUE_TEMPLATE/doc_improvement.yml, ....","[CONTRIBUTING.md, CODE_OF_CONDUCT.md]",[],"[pyproject.toml, doc/binder/requirements.txt, ...",[],"[flake8, mypy, pre-commit, ruff]"
4,artifact_5,Pandas,True,passed,8.0,5.0,[],[.github/dependabot.yml],[.github/workflows/codeql.yml],"[.github/workflows/unit-tests.yml, .github/wor...",[],1023.0,"[.github/ISSUE_TEMPLATE/feature_request.yaml, ...",[],[],"[environment.yml, pyproject.toml]",[],"[bandit, black, flake8, isort, mypy, pre-commi..."
5,artifact_6,NumPy,True,passed,10.0,5.0,[],[.github/dependabot.yml],"[.github/workflows/codeql.yml, .github/workflo...","[.github/workflows/dependency-review.yml, .git...",[],186.0,"[.github/ISSUE_TEMPLATE/feature-request.yml, ....","[CONTRIBUTING.rst, .github/CONTRIBUTING.md]",[],"[environment.yml, pyproject.toml, numpy/f2py/s...",[],"[flake8, isort, mypy, pylint, ruff]"
6,artifact_7,Matplotlib,True,passed,10.0,5.0,[SECURITY.md],[.github/dependabot.yml],[.github/workflows/codeql-analysis.yml],"[azure-pipelines.yml, tox.ini, .github/workflo...",[],134.0,"[.github/ISSUE_TEMPLATE/documentation.yml, .gi...","[.github/CONTRIBUTING.md, CODE_OF_CONDUCT.md]",[],"[environment.yml, pyproject.toml, lib/matplotl...",[],"[black, isort, mypy, pre-commit, ruff]"
7,artifact_8,Scrapy,True,passed,10.0,5.0,[SECURITY.md],[],[],"[tox.ini, .github/workflows/auto-close-llm-pr....",[tests],139.0,"[.github/ISSUE_TEMPLATE/bug_report.md, .github...","[CONTRIBUTING.md, CODE_OF_CONDUCT.md]",[],"[pyproject.toml, docs/requirements.txt]",[],"[bandit, black, flake8, isort, mypy, pre-commi..."
8,artifact_9,Flask,True,passed,7.0,5.0,[],[],[.github/workflows/zizmor.yaml],"[.github/workflows/tests.yaml, .github/workflo...",[tests],27.0,"[.github/ISSUE_TEMPLATE/bug-report.md, .github...",[],[],"[pyproject.toml, examples/tutorial/pyproject.t...",[uv.lock],"[flake8, isort, mypy, pre-commit, ruff]"
9,artifact_10,TensorFlow,True,passed,12.0,4.0,[SECURITY.md],[.github/dependabot.yml],"[.github/workflows/osv-scanner-scheduled.yml, ...","[.github/workflows/cffconvert.yml, .github/wor...",[],1678.0,[.github/ISSUE_TEMPLATE/tflite-converter-issue...,"[CONTRIBUTING.md, CODE_OF_CONDUCT.md]",[RELEASE.md],[third_party/xla/xla/backends/cpu/benchmarks/e...,[],[pylint]


In [5]:
output_file = "maintainable_security_results.json"

with open(output_file, "w", encoding="utf-8") as file:
    json.dump(maintainable_security_results, file, indent=4)

print(f"✅ Results saved to {output_file}")

✅ Results saved to maintainable_security_results.json
